# Suite di test — Classificatore malattie foglie (PlantVillage)

Notebook **dimostrativo** e separato dall'originale `models/leaves_classifier.ipynb`,
che resta intatto. Qui:

1. si importano le funzioni estratte in `src/leaves/` (copiate verbatim dal notebook originale);
2. si esegue la suite `pytest` inline mostrandone l'esito reale.

La suite è in `tests/`; questo notebook è solo l'artefatto che la mostra in azione.

> **Test ≠ valutazione.** Qui si verificano invarianti deterministiche (forme dei tensori,
> parsing, regole di dominio). Non si asserisce nulla su accuratezza / AUROC / F1: quelle sono
> metriche stocastiche, oggetto di valutazione nel notebook originale, non di test.

## 1. Import dalla logica estratta (`src/leaves/`)

In [1]:
import sys, os
sys.path.insert(0, "src")
os.environ.setdefault("KERAS_BACKEND", "torch")  # come la cella 2 del notebook originale

from leaves.data import parse_class, load_images
from leaves.models import build_encoder, build_decoder, build_autoencoder, IMG_SIZE, BOTTLENECK_CH
from leaves.pipeline import select_healthy_training, balanced_sample, encode_labels

print("Import da src/ riusciti.")
print("IMG_SIZE =", IMG_SIZE, "| BOTTLENECK_CH =", BOTTLENECK_CH)

Import da src/ riusciti.
IMG_SIZE = 128 | BOTTLENECK_CH = 32


## 2. Cosa verifica ogni gruppo di test

### `tests/test_data.py` — parsing e caricamento immagini
- `parse_class` estrae `(pianta, condizione, sana)` dal nome cartella `Pianta___Condizione`:
  gestisce le parentesi della specie (`Corn_(maize)` → `Corn`) e riconosce `healthy` in modo
  **case-insensitive**.
- `load_images` restituisce un array `(N, size, size, 3)` `float32`, con pixel in `[0, 255]` e
  ridimensionamento a quadrato anche da input non quadrato. Le immagini sono **sintetiche**
  (generate in `tmp_path`), mai il dataset reale.

### `tests/test_models.py` — smoke test delle sole forme
- `build_encoder`: `(1,128,128,3) → (1,8,8,BOTTLENECK_CH)`.
- `build_decoder`: `(1,8,8,BOTTLENECK_CH) → (1,128,128,3)`.
- `build_autoencoder`: round-trip con stessa forma in ingresso e uscita.

Proteggono l'**invariante 8×8 hardcoded** nel decoder (deriva da `IMG_SIZE=128` ridotto da 4
MaxPooling): se `IMG_SIZE` cambia, encoder e decoder non combaciano più e i test falliscono.
Nessun addestramento.

### `tests/test_pipeline.py` — invarianti di dominio
- Il set di training dell'autoencoder contiene **zero foglie malate** (presupposto non supervisionato).
- Il campionamento bilanciato non supera mai **250** esempi per classe (`min(250, len(group))`).
- `LabelEncoder`: `encode` + `inverse_transform` riporta alle etichette originali.

## 3. Esecuzione della suite `pytest`

Esecuzione reale con coverage su `src/leaves` (configurazione in `pyproject.toml`).

In [2]:
!"{sys.executable}" -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.14.0, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\MarcoProcopio\Downloads\python-string-utils\Progetto ML\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\MarcoProcopio\Downloads\python-string-utils\Progetto ML
configfile: pyproject.toml
testpaths: tests
plugins: cov-7.1.0
collecting ... collected 12 items

tests/test_data.py::test_parse_class_healthy PASSED                      [  8%]
tests/test_data.py::test_parse_class_malata PASSED                       [ 16%]
tests/test_data.py::test_parse_class_pulisce_parentesi PASSED            [ 25%]
tests/test_data.py::test_parse_class_healthy_case_insensitive PASSED     [ 33%]
tests/test_data.py::test_load_images_forma_e_dtype PASSED                [ 41%]
tests/test_data.py::test_load_images_valori_in_range_e_resize_da_non_quadrato PASSED [ 50%]
tests/test_models.py::test_encoder_output_shape PASSED                   [ 58%]